# Module 2 · Chapter 3 — 독립성 함정

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 독립의 수식 정의, 독립 검증, 조건부 독립
- **이 노트북**: 운동·수면 데이터로 독립 가정을 직접 검증

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. 시나리오 데이터 생성

두 버전을 만들어 비교합니다.
- **독립 버전**: 운동 여부와 수면 품질이 실제로 독립
- **의존 버전**: 운동 많이 하는 사람이 수면 품질도 높은 경향

In [ ]:
n = 1000
p_exercise = 0.40  # 운동 규칙적(Yes) 비율
p_good_sleep = 0.55  # 수면 Good 비율

# --- 독립 버전 ---
exercise_ind = rng.random(n) < p_exercise
sleep_ind    = rng.random(n) < p_good_sleep

# --- 의존 버전 ---
# 운동하면 수면 Good 확률 0.72, 안 하면 0.44
exercise_dep = rng.random(n) < p_exercise
sleep_dep = np.where(
    exercise_dep,
    rng.random(n) < 0.72,
    rng.random(n) < 0.44
)

df_ind = pd.DataFrame({"운동": exercise_ind, "수면Good": sleep_ind})
df_dep = pd.DataFrame({"운동": exercise_dep, "수면Good": sleep_dep})

print("=== 독립 버전 ===")
print(pd.crosstab(df_ind["운동"], df_ind["수면Good"], normalize="all").round(3))
print("\n=== 의존 버전 ===")
print(pd.crosstab(df_dep["운동"], df_dep["수면Good"], normalize="all").round(3))

## 2. 직관 — P(A∩B) vs P(A)×P(B) 비교

독립이면: P(운동 AND 수면Good) = P(운동) × P(수면Good)

이 등식이 성립하는지 두 버전에서 각각 확인합니다.

In [ ]:
def independence_check(df, label):
    p_a   = df["운동"].mean()
    p_b   = df["수면Good"].mean()
    p_a_b = (df["운동"] & df["수면Good"]).mean()
    p_independent = p_a * p_b

    print(f"[{label}]")
    print(f"  P(운동)          = {p_a:.3f}")
    print(f"  P(수면Good)      = {p_b:.3f}")
    print(f"  P(A) × P(B)     = {p_independent:.3f}  ← 독립 가정")
    print(f"  P(A ∩ B) 실제   = {p_a_b:.3f}  ← 데이터에서 직접")
    print(f"  차이             = {abs(p_a_b - p_independent):.4f}")
    print(f"  독립으로 볼 수 있는가? {'≈ YES' if abs(p_a_b - p_independent) < 0.01 else '✗ NO — 의존성 있음'}")
    print()

independence_check(df_ind, "독립 버전")
independence_check(df_dep, "의존 버전")

## 3. 손으로 계산 — 조건부 확률로 독립 검증

독립 조건: P(수면Good|운동=Yes) = P(수면Good)

In [ ]:
for df, label in [(df_ind, "독립 버전"), (df_dep, "의존 버전")]:
    p_sleep = df["수면Good"].mean()
    p_sleep_given_exercise    = df[df["운동"]]["수면Good"].mean()
    p_sleep_given_no_exercise = df[~df["운동"]]["수면Good"].mean()

    print(f"[{label}]")
    print(f"  P(수면Good)              = {p_sleep:.3f}")
    print(f"  P(수면Good | 운동=Yes)   = {p_sleep_given_exercise:.3f}")
    print(f"  P(수면Good | 운동=No)    = {p_sleep_given_no_exercise:.3f}")
    print()

## 4. 라이브러리로 재현 — 히트맵 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df, title in zip(
    axes,
    [df_ind, df_dep],
    ["독립 버전", "의존 버전"]
):
    ct = pd.crosstab(
        df["운동"].map({True: "운동 Yes", False: "운동 No"}),
        df["수면Good"].map({True: "수면 Good", False: "수면 Poor"}),
        normalize="all"
    )
    sns.heatmap(ct, annot=True, fmt=".3f", cmap="Blues",
                vmin=0, vmax=0.4, ax=ax, cbar=False)
    ax.set_title(title, fontsize=12)

plt.suptitle("교차표 비교: 독립이면 행·열이 비례해야 합니다", fontsize=13)
plt.tight_layout()
plt.show()

## 5. 함정 점검 — 독립처럼 보이지만 아닌 경우

전체적으로는 독립처럼 보여도, 연령대를 조건으로 하면 의존성이 나타나는 사례.

In [ ]:
# 연령대: 젊음(0) = 운동 많이, 수면 좋음 / 고령(1) = 운동 적게, 수면 나쁨
age = rng.choice([0, 1], size=n, p=[0.5, 0.5])  # 0=젊음, 1=고령

# 연령에 따른 운동·수면 → 전체적으로는 독립처럼 보이게 설계
exercise_confounded = np.where(
    age == 0,
    rng.random(n) < 0.65,  # 젊음: 운동 많음
    rng.random(n) < 0.20   # 고령: 운동 적음
)
sleep_confounded = np.where(
    age == 0,
    rng.random(n) < 0.70,  # 젊음: 수면 좋음
    rng.random(n) < 0.35   # 고령: 수면 나쁨
)

df_conf = pd.DataFrame({"운동": exercise_confounded, "수면Good": sleep_confounded, "연령": age})

print("=== 전체 데이터 (연령 무시) ===")
p_a  = df_conf["운동"].mean()
p_b  = df_conf["수면Good"].mean()
p_ab = (df_conf["운동"] & df_conf["수면Good"]).mean()
print(f"P(A)×P(B) = {p_a*p_b:.3f},  P(A∩B) = {p_ab:.3f}  → {'독립처럼 보임' if abs(p_ab - p_a*p_b) < 0.02 else '의존'}")

print()
print("=== 연령대별 조건부 독립 확인 ===")
for age_val, label in [(0, "젊음"), (1, "고령")]:
    sub = df_conf[df_conf["연령"] == age_val]
    pa  = sub["운동"].mean()
    pb  = sub["수면Good"].mean()
    pab = (sub["운동"] & sub["수면Good"]).mean()
    print(f"  [{label}] P(A)×P(B) = {pa*pb:.3f},  P(A∩B) = {pab:.3f}  → 독립?")

## 6. 직접 답해 보기

1. 두 변수가 독립인지 확인하는 세 가지 방법(P(A∩B) 비교, 조건부 확률, 교차표)을 하나씩 설명한다면?
2. 위 혼동 시뮬레이션에서 연령을 고려하지 않으면 어떤 오류 결론을 내릴 수 있었을까?
3. 내가 관심 있는 두 변수를 골라보자. 그 둘 사이의 독립 가정을 어떻게 검증할 수 있을까?